# Fine-tuning the small reasoner: Gemma 3 4B with LoRA (Item 4, DEC-30)

DEC-30 (17 Nov 2026, amended): Gemma 3 4B instruction-tuned is fine-tuned with LoRA (bf16, on this A100) on the gold answers of EXP-43, 1,080 synthetic training scenarios and 120 for validation, in prompt v3's exact request format. The loss is on the answer only. Logic in `src/strokecopilot/nlp/reasoner/finetune.py`. No Anthropic or OpenRouter key is used here, and no patient data: the scenarios are synthetic.

- `MODE = 'dummy'` (EXP-44): 30 optimiser steps on 20 examples, then one generated answer. It reports peak GPU memory, tokens per second, the projected time of the full run, and whether the answer is valid JSON for the schema. About 15–20 minutes (estimate).
- `MODE = 'run'` (EXP-45): the full fine-tune with early stopping on the validation loss, then the best adapter merged into a full model on Drive for serving. Hours: the dummy run gives the estimate.

**Before you start**
- Runtime → Change runtime type → **A100 GPU**.
- On Hugging Face, accept the terms of `google/gemma-3-4b-it`. In Secrets (the key icon), `HF_TOKEN` must have notebook access on. `HF_TOKEN` is a fine-grained, read-only token for `google/gemma-3-4b-it`. The token is not printed or kept in a variable after use.
- Keep this tab open while it runs.

**Run it:** Runtime → Run all, and allow Drive access. **If Colab disconnects** during `run`, reconnect and Run all again: training resumes from the last checkpoint on Drive. Done when the last cell prints `OK:`.

## 1. Settings

In [ ]:
MODE = 'dummy'   # 'dummy' (EXP-44) or 'run' (EXP-45)
BALANCED = False   # True: DEC-39, post-freeze, report only: train on EXP-91 (decision-balanced) as EXP-92
EXPERIMENT, NAME = ('EXP-44', 'EXP-44_finetune_dummy') if MODE == 'dummy' else ('EXP-45', 'EXP-45_finetune')
DATA_EXP, DATA_NAME = 'EXP-43', 'EXP-43_train_set'
if BALANCED:
    EXPERIMENT, NAME = 'EXP-92', 'EXP-92_finetune_balanced' + ('_dummy' if MODE == 'dummy' else '')
    DATA_EXP, DATA_NAME = 'EXP-91', 'EXP-91_train_set_balanced'
ROOT = '/content/drive/MyDrive/StrokeCopilot'
REPO_REF = 'main'
DATA = f'{ROOT}/runs/{DATA_NAME}'   # the training set (train.jsonl, val.jsonl, system_prompt.md, meta.json)
RUN_DIR = f'{ROOT}/runs/{NAME}'
TEST_SET_SHA256 = 'ea5b5977cd8e0bf78cb8929f3b1ab61f3ba512944fd21ffa94d30c65b5112e42'   # the 150 evaluation scenarios
print(f'{EXPERIMENT} ({MODE}, data {DATA_EXP}): output to {RUN_DIR}.')

## 2. Install the training libraries and the package
The package is pinned to the commit `REPO_REF` points at, with its `finetune` extra (transformers, peft and accelerate at the versions the CPU tests used).

In [ ]:
import json, os, subprocess, sys, time
from google.colab import drive, userdata

if any(m == 'strokecopilot' or m.startswith('strokecopilot.') for m in sys.modules):
    raise RuntimeError('This session already loaded an older copy of the package. Use Runtime → Restart session, then Run all.')
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
if 'A100' not in gpu:
    raise RuntimeError(f'This runtime has {gpu or "no GPU"}. Runtime → Change runtime type → A100 GPU, then Run all.')
print('GPU:', gpu)
started = time.time()
repo = 'https://github.com/parsa-hke/strokecopilot.git'
found = subprocess.run(['git', 'ls-remote', repo, REPO_REF], capture_output=True, text=True)
if found.returncode or not found.stdout.strip():
    raise RuntimeError(('Could not read the repo. ' + found.stderr)[-800:])
COMMIT = found.stdout.split()[0]
for args in ([f'strokecopilot[finetune] @ git+{repo}@{COMMIT}'], ['--force-reinstall', '--no-deps', f'strokecopilot @ git+{repo}@{COMMIT}']):
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(result.stderr[-800:])
del repo, found, result
# Colab ships torchao 0.10, which peft 0.21 refuses to import alongside (it needs >0.16). LoRA here does not use torchao,
# so it is removed rather than upgraded (an upgrade could pull a different torch).
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], capture_output=True, text=True)
import importlib.util
if importlib.util.find_spec('torchao') is not None:
    raise RuntimeError('torchao is still installed and peft will refuse it. Runtime → Restart session, then Run all.')
import torch, transformers, peft
print(f'torch {torch.__version__}, transformers {transformers.__version__}, peft {peft.__version__}; package at commit {COMMIT[:7]}; installed in {time.time() - started:.0f} s.')

## 3. Drive and the training set
The training set is copied to this machine's disk and checked: it must be EXP-43, built against the unchanged evaluation set.

In [ ]:
import shutil
drive.mount('/content/drive')
LOCAL = '/content/train_set'
shutil.copytree(DATA, LOCAL, dirs_exist_ok=True)
from strokecopilot.nlp.reasoner import finetune as ft
meta = ft.check_dataset(LOCAL, DATA_EXP, TEST_SET_SHA256)   # raises unless DATA_EXP, built against the fixed evaluation set, files unchanged
system = open(f'{LOCAL}/system_prompt.md', encoding='utf-8').read()
train_rows, val_rows = ft.read_jsonl(f'{LOCAL}/train.jsonl'), ft.read_jsonl(f'{LOCAL}/val.jsonl')
if MODE == 'dummy':
    train_rows, val_rows = train_rows[:20], val_rows[:4]
os.makedirs(RUN_DIR, exist_ok=True)
print(f'{len(train_rows)} training and {len(val_rows)} validation examples; held-out routes: {meta["held_out_routes"]}')

## 4. Load Gemma 3 4B and add the LoRA adapter
The Hugging Face token is passed to the downloads only and deleted after them. The whole base repo is downloaded here, so the merge after training reads it from the cache without a token.

In [ ]:
if MODE == 'dummy':
    config = ft.FinetuneConfig(max_steps=30, grad_accum=1, eval_every=10, warmup_steps=5, patience=100)
else:
    config = ft.FinetuneConfig()
hf = userdata.get('HF_TOKEN')
model, tokenizer = ft.load_base(config.model_id, token=hf)
ft.prefetch(config.model_id, token=hf)   # the whole repo into the cache, so the merge after training needs no token
del hf
model = ft.lora_model(model, config)
model.print_trainable_parameters()
path = f'{RUN_DIR}/config.json'
if not os.path.exists(path):
    with open(path, 'w') as f:
        json.dump({'experiment': EXPERIMENT, 'mode': MODE, 'commit': COMMIT, 'gpu': gpu, 'data': DATA_EXP, 'data_meta': meta,
                   'versions': {'torch': torch.__version__, 'transformers': transformers.__version__, 'peft': peft.__version__},
                   'finetune': ft.asdict(config), 'started_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}, f, indent=1)

## 5. Train
In `dummy` mode: 30 steps, then one generated answer, and the projected time of the full run. In `run` mode: the full fine-tune (resuming if a checkpoint exists), then the best adapter merged into a full model for serving, and three validation answers from it checked: each must stop by itself and validate.

In [ ]:
if MODE == 'dummy':
    report = ft.dummy(model, tokenizer, system, train_rows, val_rows, RUN_DIR, config)
    full = ft.FinetuneConfig()
    per_epoch_tokens = meta['train'] * report['tokens_per_example']['mean']
    report['projected_full_run_hours'] = round(full.epochs * per_epoch_tokens / report['tokens_per_s'] / 3600, 2) if report['tokens_per_s'] else None
    json.dump(report, open(f'{RUN_DIR}/dummy_report.json', 'w'), indent=1)
    print(json.dumps(report, indent=1))
else:
    record = ft.train(model, tokenizer, system, train_rows, val_rows, RUN_DIR, config)
    print(f"Trained {record['steps']} steps; best validation loss {record['best_val_loss']:.4f}.")
    del model
    torch.cuda.empty_cache()
    # From the cache, with no token: Colab may not hand over a secret hours into a run (EXP-45's first merge failed so).
    merged = ft.merge(config.model_id, f'{RUN_DIR}/best', f'{RUN_DIR}/merged', local_files_only=True)
    print('Merged model saved to', merged)
    # Validation loss cannot see a model that loops, so the merged (best) model writes three validation answers.
    tuned, tuned_tok = ft.load_base(str(merged))
    check = ft.generation_check(tuned, tuned_tok, system, val_rows, RUN_DIR)
    print(json.dumps(check, indent=1))
    del tuned
    torch.cuda.empty_cache()

## 6. Summary

In [ ]:
if MODE == 'dummy':
    ok = report['generated_valid'] and report['generated_stopped'] and report['tokens_per_s']
    print((f"OK: {EXPERIMENT} at {COMMIT[:7]}: " if ok else 'CHECK: ') +
          f"{report['steps']} steps in {report['seconds']} s, peak GPU {report['peak_gpu_gb']} GB, {report['tokens_per_s']:.0f} tokens/s, "
          f"{report['tokens_per_example']['mean']:.0f} tokens per example; validation loss {report['val_loss']}; "
          f"generated answer valid: {report['generated_valid']} {report['generated_problem'] or ''}; "
          f"{report['generated_tokens']} tokens, stopped by itself: {report['generated_stopped']} (full text in generated.txt); "
          f"projected full run {report['projected_full_run_hours']} h.")
else:
    files = sorted(os.listdir(f'{RUN_DIR}/merged'))
    ok = check['stopped'] == check['valid'] == check['n']
    print((f"OK: {EXPERIMENT} at {COMMIT[:7]}: " if ok else 'CHECK: ') +
          f"{record['steps']} steps, best validation loss {record['best_val_loss']:.4f}; merged model files: {len(files)}; "
          f"of {check['n']} validation answers, {check['stopped']} stopped by themselves, {check['valid']} valid, "
          f"{check['verdicts_match_gold']} with the gold verdicts (answers in generated_*.txt).")